# Phase 2 — Data v0: the bug factory meets real functions

**Runtime: plain CPU** (zero compute units). ~15–30 min. Run all.

**What this does:** feeds MBPP+'s 378 verified-correct Python functions through the
repo's mutation factory (`src/mutate.py`) and produces **data v0**:
- certified training bugs (each compiles AND fails ≥1 test — proven by execution)
- ≤2 bugs per function, distinct categories (diversity rule)
- function-level train/dev/heldout splits (mutants never straddle splits)
- contamination screen against the 164 exam problems (name + solution overlap)
- the restraint suite for free (verified-clean originals)

**v0 honesty note:** the mutant-validation gate uses MBPP's original assertions
(`test_list`, 3–7 per problem). Conservative direction: weak tests can only cost us
yield (real bugs wrongly discarded as 'equivalent'), never let a fake bug through.
Integrating the stronger MBPP+ extended suites is a flagged v1 upgrade.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PHASE2 = '/content/drive/MyDrive/rl-post-training/phase2'
os.makedirs(PHASE2, exist_ok=True)
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl 2>/dev/null || git -C /content/ptl pull -q
!pip install -q datasets
import sys
sys.path.insert(0, '/content/ptl/src')
sys.path.insert(0, '/content/ptl/eval')
from mutate import generate_mutants, select_mutants, assign_split
print('Factory loaded. Output ->', PHASE2)

In [ ]:
# Load the raw material (MBPP+) and the exam (for the contamination screen)
from datasets import load_dataset
mbpp = load_dataset('evalplus/mbppplus', split='test')
exam = load_dataset('bigcode/humanevalpack', 'python', split='test')
print(len(mbpp), 'candidate functions |', len(exam), 'exam problems')

In [ ]:
# Contamination screen: drop any function whose NAME collides with an exam
# entry point, or whose normalized solution matches an exam solution.
import ast

def def_names(src):
    try:
        return [n.name for n in ast.walk(ast.parse(src)) if isinstance(n, ast.FunctionDef)]
    except SyntaxError:
        return None

def normalize(src):
    try:
        return ast.unparse(ast.parse(src))
    except SyntaxError:
        return None

exam_names = {row['entry_point'] for row in exam}
exam_solutions = {normalize(row['prompt'] + row['canonical_solution']) for row in exam}

problems, dropped = [], []
for row in mbpp:
    names = def_names(row['code'])
    if not names:
        dropped.append((row['task_id'], 'unparseable')); continue
    if set(names) & exam_names:
        dropped.append((row['task_id'], f'name collision: {set(names) & exam_names}')); continue
    if normalize(row['code']) in exam_solutions:
        dropped.append((row['task_id'], 'solution overlap')); continue
    problems.append(row)
print(f'{len(problems)} functions pass the screen; {len(dropped)} dropped:')
for tid, why in dropped[:20]:
    print(f'  Mbpp/{tid}: {why}')

In [ ]:
# Execution helpers: run code+tests in a subprocess with a timeout
import subprocess, sys as _sys, tempfile

def run_with_tests(source, row, timeout=6):
    prog = '\n'.join(list(row['test_imports']) + [source] + list(row['test_list']))
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(prog); path = f.name
    try:
        r = subprocess.run([_sys.executable, path], capture_output=True, timeout=timeout)
        return 'pass' if r.returncode == 0 else 'fail'
    except subprocess.TimeoutExpired:
        return 'timeout'
    finally:
        os.unlink(path)

# GOLD-SANITY GATE: the original must pass its own tests, else we refuse to mutate it
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=4) as pool:
    verdicts = list(pool.map(lambda row: run_with_tests(row['code'], row), problems))
gold = [row for row, v in zip(problems, verdicts) if v == 'pass']
print(f'Gold-sanity gate: {len(gold)}/{len(problems)} originals pass their own tests')
print(f'  (refused to mutate {len(problems)-len(gold)} bad-gold problems)')

In [ ]:
# THE FACTORY RUN: generate candidates, certify by execution, keep <=2 diverse bugs
from collections import Counter, defaultdict

def certify(row, budget=3, max_per_cat=4):
    """Round-robin across categories, validating until `budget` real bugs found."""
    stats = Counter()
    try:
        cands = generate_mutants(row['code'], max_per_category=max_per_cat)
    except SyntaxError:
        return [], stats
    by_cat = defaultdict(list)
    for m in cands:
        by_cat[m.category].append(m)
    valid = []
    while len(valid) < budget and any(by_cat.values()):
        for cat in sorted(by_cat):
            if len(valid) >= budget or not by_cat[cat]:
                continue
            m = by_cat[cat].pop(0)
            try:
                compile(m.source, '<mutant>', 'exec')
            except (SyntaxError, ValueError):
                stats['broken'] += 1; continue
            v = run_with_tests(m.source, row)
            if v == 'fail':
                stats['valid'] += 1; valid.append(m)
            elif v == 'timeout':
                stats['timeout_bug'] += 1; valid.append(m)  # infinite loop IS a bug
            else:
                stats['equivalent'] += 1
    return valid, stats

records, totals = [], Counter()
for i, row in enumerate(gold):
    valid, stats = certify(row)
    totals.update(stats)
    picked = select_mutants(valid, k=2, seed=int(row['task_id']))
    split = assign_split(row['code'])
    for j, m in enumerate(picked):
        records.append({
            'id': f"Mbpp/{row['task_id']}-{m.category}-{j}",
            'source_task': f"Mbpp/{row['task_id']}",
            'split': split,
            'category': m.category,
            'description': m.description,
            'buggy': m.source,
            'fixed': row['code'],
            'test_imports': list(row['test_imports']),
            'test_list': list(row['test_list']),
        })
    if (i + 1) % 25 == 0:
        print(f'{i+1}/{len(gold)} functions processed, {len(records)} bugs so far')
print('Factory done.')

In [ ]:
# THE REPORT — the numbers that gate Phase 2
import json
print('=== CERTIFICATION ===')
checked = sum(totals.values())
for k in ('valid', 'equivalent', 'broken', 'timeout_bug'):
    print(f'  {k:<12} {totals[k]:>5}  ({totals[k]/max(checked,1)*100:.1f}% of {checked} checked)')
print(f'\n=== KEPT: {len(records)} certified bugs ===')
print('by category:')
for cat, n in Counter(r['category'] for r in records).most_common():
    print(f'  {cat:<18} {n:>4}')
print('by split:')
for sp, n in Counter(r['split'] for r in records).most_common():
    print(f'  {sp:<18} {n:>4}')
print('\nsource functions contributing bugs:', len({r['source_task'] for r in records}))

In [ ]:
# SAVE: the bug corpus + the free restraint suite (verified-clean originals)
restraint = [{
    'id': f"Mbpp/{row['task_id']}-clean",
    'source_task': f"Mbpp/{row['task_id']}",
    'split': assign_split(row['code']),
    'code': row['code'],
    'test_imports': list(row['test_imports']),
    'test_list': list(row['test_list']),
} for row in gold]

json.dump(records, open(f'{PHASE2}/data_v0_bugs.json', 'w'), indent=1)
json.dump(restraint, open(f'{PHASE2}/data_v0_restraint.json', 'w'), indent=1)
json.dump([{'task_id': t, 'reason': r} for t, r in dropped],
          open(f'{PHASE2}/contamination_drops.json', 'w'), indent=1)
print('Saved to Drive/phase2: data_v0_bugs.json, data_v0_restraint.json, contamination_drops.json')
print('\n=== ONE SAMPLE BUG (read it!) ===')
ex = records[0]
print(f"[{ex['id']}] {ex['description']} (split={ex['split']})")
print('--- buggy ---'); print(ex['buggy'][:800])
print('--- tests ---'); print('\n'.join(ex['test_list'][:3]))

## Bring back to the session
1. The **certification table** (valid / equivalent / broken counts)
2. The **kept-bugs count + category and split histograms**
3. The **contamination drop list** (how many, and the reasons)

Next after this: the GPU routing pass (sample the base model on every training bug →
all-fail bugs weight into the SFT set, mixed-outcome into the RL set — Amendment A2).